# Baseline Model Comparison

Compares the baseline models from `src/train_baselines.py` on the **same data, location split and metrics**. The results are saved to `results/baselines.json`.

This notebook doesn't train models itself; the script is the single source of the numbers. Set `RERUN_BASELINES = True` to regenerate the results (takes about 1.5 minutes).

**Compare models on the test columns.** The validation columns are for reference.

In [1]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()
BASELINE_PATH = ROOT / "results" / "baselines.json"

RERUN_BASELINES = False   # True = retrain all baselines and overwrite results/baselines.json

if RERUN_BASELINES or not BASELINE_PATH.exists():
    print("Training baselines...")
    subprocess.run([sys.executable, "-m", "src.train_baselines", "--output", str(BASELINE_PATH)],
                   cwd=ROOT, check=True)

baselines = json.loads(BASELINE_PATH.read_text())

## Data and split used

`data_sha256` is a fingerprint of the data file, and `split_check` confirms the split matches `docs/region_split_seed42.json`. Any later model has to match these to be compared fairly.

In [2]:
pd.Series({
    "data_sha256": baselines["data_sha256"],
    "split_seed": baselines["split_seed"],
    "split_check": baselines["split_check"],
    "rows (train / val / test)": " / ".join(
        str(baselines["split"][p]["rows"]) for p in ("train", "validation", "test")),
})

data_sha256                  dad983e4a1701ef474b11f51adcea94f7ddd35ca23d6d5...
split_seed                                                                  42
split_check                              matches docs/region_split_seed42.json
rows (train / val / test)                                  27973 / 7553 / 8035
dtype: object

## Collect the results into one table

One row per model, feature set, split and metric.

In [3]:
FEATURES = {"model1_weather": "1. Weather",
            "model2_weather_veg": "2. + Vegetation",
            "model3_all": "3. + Geography"}

rows = []
for task in ("regression", "classification"):
    for group, models in baselines[task].items():
        for model, scores in models.items():
            for split, metrics in scores.items():
                for metric, value in metrics.items():
                    if isinstance(value, dict):   # per-class results and confusion matrix: see their own section
                        continue
                    rows.append(dict(task=task, features=FEATURES[group], model=model,
                                     split=split, metric=metric, value=value))
all_results = pd.DataFrame(rows)


def comparison(task, metric, split="test"):
    """Models as rows, feature sets as columns."""
    d = all_results.query("task == @task and metric == @metric and split == @split")
    model_order = list(dict.fromkeys(d["model"]))
    return d.pivot(index="model", columns="features", values="value").loc[model_order].round(3)


def best_model(task, metric, higher_is_better, split="test"):
    """The best non-dummy model for each feature set."""
    d = all_results.query("task == @task and metric == @metric and split == @split "
                          "and not model.str.startswith('dummy')")
    grouped = d.groupby("features")["value"]
    idx = grouped.idxmax() if higher_is_better else grouped.idxmin()
    return d.loc[idx, ["features", "model", "value"]].rename(columns={"value": metric}).round(3).set_index("features")


def all_metrics(task):
    """Every metric for every model, validation and test side by side."""
    return (all_results.query("task == @task")
            .pivot_table(index=["features", "model"], columns=["split", "metric"], values="value", sort=False)
            .round(3))


all_results.head()

,task,features,model,split,metric,value
0,regression,1. Weather,dummy_mean,validation,rmse_log,1.164338
1,regression,1. Weather,dummy_mean,validation,mae_log,0.663559
2,regression,1. Weather,dummy_mean,validation,r2_log,-0.000315
3,regression,1. Weather,dummy_mean,validation,rmse_acres,1185.620994
4,regression,1. Weather,dummy_mean,validation,mae_acres,62.972975


## Regression: predicting fire size (log acres)
We are primarily focusing on classification so this part is mostly trivial.

Metrics come from `src/evaluate.py`:
- `rmse_log`, `mae_log`: errors on the log scale. **This means that lower is better.** `rmse_log` is the main metric.
- `r2_log`: share of variation explained. **This means that higher is better.** 0 is the same as always predicting the average.
- `rmse_acres`, `mae_acres`: errors after converting back to acres. A few huge fires dominate these.

`dummy_mean` (always predict the training average) is the floor every model has to beat.

### Test log RMSE (main comparison, lower is better)

In [4]:
comparison("regression", "rmse_log")

features,1. Weather,2. + Vegetation,3. + Geography
model,,,
dummy_mean,1.178,1.178,1.178
dummy_median,1.267,1.267,1.267
linear_regression,1.163,1.135,1.123
random_forest,1.171,1.144,1.129


### Test R² on the log scale (higher is better)

In [5]:
comparison("regression", "r2_log")

features,1. Weather,2. + Vegetation,3. + Geography
model,,,
dummy_mean,-0.000,-0.000,-0.000
dummy_median,-0.157,-0.157,-0.157
linear_regression,0.024,0.071,0.090
random_forest,0.011,0.057,0.081


### Best regression model for each feature set

In [6]:
best_model("regression", "rmse_log", higher_is_better=False)

,model,rmse_log
features,,
1. Weather,linear_regression,1.163
2. + Vegetation,linear_regression,1.135
3. + Geography,linear_regression,1.123


### All regression metrics, validation and test

In [7]:
all_metrics("regression")

split                             validation                            \
metric                              rmse_log mae_log r2_log rmse_acres   
features        model                                                    
1. Weather      dummy_mean             1.164   0.664 -0.000   1185.621   
                dummy_median           1.267   0.542 -0.184   1185.657   
                linear_regression      1.155   0.670  0.015   1185.598   
                random_forest          1.169   0.715 -0.008   1185.563   
2. + Vegetation dummy_mean             1.164   0.664 -0.000   1185.621   
                dummy_median           1.267   0.542 -0.184   1185.657   
                linear_regression      1.136   0.663  0.047   1185.575   
                random_forest          1.150   0.694  0.025   1185.555   
3. + Geography  dummy_mean             1.164   0.664 -0.000   1185.621   
                dummy_median           1.267   0.542 -0.184   1185.657   
                linear_regression      1.132   0.654  0.054   1185.570   
                random_forest          1.138   0.691  0.044   1185.503   

split                                           test                 \
metric                            mae_acres rmse_log mae_log r2_log   
features        model                                                 
1. Weather      dummy_mean           62.973    1.178   0.660 -0.000   
                dummy_median         62.791    1.267   0.509 -0.157   
                linear_regression    62.999    1.163   0.658  0.024   
                random_forest        63.148    1.171   0.705  0.011   
2. + Vegetation dummy_mean           62.973    1.178   0.660 -0.000   
                dummy_median         62.791    1.267   0.509 -0.157   
                linear_regression    63.009    1.135   0.636  0.071   
                random_forest        63.155    1.144   0.676  0.057   
3. + Geography  dummy_mean           62.973    1.178   0.660 -0.000   
                dummy_median         62.791    1.267   0.509 -0.157   
                linear_regression    63.000    1.123   0.635  0.090   
                random_forest        63.146    1.129   0.675  0.081   

split                                                   
metric                            rmse_acres mae_acres  
features        model                                   
1. Weather      dummy_mean          5113.615   197.881  
                dummy_median        5113.641   197.657  
                linear_regression   5113.589   197.895  
                random_forest       5113.555   198.057  
2. + Vegetation dummy_mean          5113.615   197.881  
                dummy_median        5113.641   197.657  
                linear_regression   5113.582   197.867  
                random_forest       5113.545   198.043  
3. + Geography  dummy_mean          5113.615   197.881  
                dummy_median        5113.641   197.657  
                linear_regression   5113.562   197.871  
                random_forest       5113.481   198.041

## Classification: predicting the fire size class (A, B, C, D-E, F-G)

- `macro_f1`: average F1 across the five classes, so the rare large-fire classes count as much as small fires. **Main metric.**
- `balanced_accuracy`: average recall per class, or how many of fires of each class that the model caught. It does not penalize the model for labeling fires for classes that they do not belong to. 
- `accuracy`: this is a little bit misleading as always guessing class A gets about 0.61–0.65.

`dummy_majority` (always guess the most common class) is the floor.

### Test macro F1 (main comparison, higher is better)
Macro F1 calculates the F1 separately for each of the 5 classes, and then takes the average. F1 combines the how often the model correctly predicts an F-G fire, and how many of the real F-G fires that the model catches. These are inverses, so you want to maximize the value. 

In [8]:
comparison("classification", "macro_f1")

features,1. Weather,2. + Vegetation,3. + Geography
model,,,
dummy_majority,0.157,0.157,0.157
logistic,0.176,0.208,0.212
random_forest,0.242,0.266,0.290


### Test balanced accuracy (higher is better)

In [9]:
comparison("classification", "balanced_accuracy")

features,1. Weather,2. + Vegetation,3. + Geography
model,,,
dummy_majority,0.200,0.200,0.200
logistic,0.343,0.352,0.344
random_forest,0.257,0.287,0.321


### Best classification model for each feature set

In [10]:
best_model("classification", "macro_f1", higher_is_better=True)

,model,macro_f1
features,,
1. Weather,random_forest,0.242
2. + Vegetation,random_forest,0.266
3. + Geography,random_forest,0.290


### All classification metrics, validation and test

In [11]:
all_metrics("classification")

split                          validation                                test  \
metric                           accuracy balanced_accuracy macro_f1 accuracy   
features        model                                                           
1. Weather      dummy_majority      0.613             0.200    0.152    0.648   
                logistic            0.261             0.276    0.166    0.284   
                random_forest       0.496             0.251    0.241    0.501   
2. + Vegetation dummy_majority      0.613             0.200    0.152    0.648   
                logistic            0.327             0.318    0.197    0.363   
                random_forest       0.506             0.265    0.254    0.517   
3. + Geography  dummy_majority      0.613             0.200    0.152    0.648   
                logistic            0.367             0.314    0.214    0.380   
                random_forest       0.503             0.269    0.256    0.535   

split                                                      
metric                         balanced_accuracy macro_f1  
features        model                                      
1. Weather      dummy_majority             0.200    0.157  
                logistic                   0.343    0.176  
                random_forest              0.257    0.242  
2. + Vegetation dummy_majority             0.200    0.157  
                logistic                   0.352    0.208  
                random_forest              0.287    0.266  
3. + Geography  dummy_majority             0.200    0.157  
                logistic                   0.344    0.212  
                random_forest              0.321    0.290

## Classification results for each size class

The overall scores above hide how differently the models do on each class. These tables use the **test set** and, by default, **all features** (`model3_all`).

- `precision`: when the model predicts this class, how often it is right
- `recall`: of the real fires in this class, how many the model caught
- `f1`: combines precision and recall
- `support`: how many real fires of this class are in the test set

In [12]:
def per_class(model, group="model3_all", split="test"):
    """Precision, recall, F1 and support for each class. Classes are columns."""
    scores = baselines["classification"][group][model][split]["per_class"]
    table = pd.DataFrame(scores).round(2).astype(object)   # object so counts can show as whole numbers
    table.loc["support"] = table.loc["support"].astype(int)
    return table


def per_class_f1(group="model3_all", split="test"):
    """F1 for each class, with one row per model."""
    models = baselines["classification"][group]
    return pd.DataFrame({m: {c: s["f1"] for c, s in models[m][split]["per_class"].items()}
                         for m in models}).T.round(2)


def confusion(model, group="model3_all", split="test"):
    """Rows are the true class, columns the predicted class. The diagonal is correct predictions."""
    cm = baselines["classification"][group][model][split]["confusion_matrix"]
    return pd.DataFrame(cm["matrix"],
                        index=[f"true {c}" for c in cm["labels"]],
                        columns=[f"pred {c}" for c in cm["labels"]])

### F1 for each class, every model (test, all features)

In [13]:
per_class_f1()

,A,B,C,D-E,F-G
dummy_majority,0.79,0.00,0.00,0.00,0.00
logistic,0.57,0.26,0.12,0.06,0.05
random_forest,0.66,0.41,0.18,0.09,0.10


### Logistic regression: precision, recall and F1 for each class

In [14]:
per_class("logistic")

,A,B,C,D-E,F-G
precision,0.75,0.34,0.08,0.03,0.03
recall,0.46,0.21,0.32,0.29,0.44
f1,0.57,0.26,0.12,0.06,0.05
support,5210,2336,321,107,61


### Random forest: precision, recall and F1 for each class

In [15]:
per_class("random_forest")

,A,B,C,D-E,F-G
precision,0.75,0.37,0.14,0.09,0.07
recall,0.59,0.47,0.27,0.1,0.16
f1,0.66,0.41,0.18,0.09,0.1
support,5210,2336,321,107,61


### Confusion matrices (test, all features)

Each row is the true class and each column is what the model predicted, so the diagonal counts correct predictions. Read across a row to see where a class's fires ended up. For example, the `true F-G` row shows how many of the largest fires were predicted as each class.

#### Logistic regression

In [16]:
confusion("logistic")

,pred A,pred B,pred C,pred D-E,pred F-G
true A,2393,939,744,533,601
true B,728,497,448,354,309
true C,38,35,102,72,74
true D-E,12,4,29,31,31
true F-G,2,1,8,23,27


#### Random forest

In [17]:
confusion("random_forest")

,pred A,pred B,pred C,pred D-E,pred F-G
true A,3080,1728,276,63,63
true B,925,1108,219,40,44
true C,84,127,87,7,16
true D-E,23,42,29,11,2
true F-G,12,12,19,8,10


## Takeaways
